# SD3.5 inpaint-EDIT LoRA — all-in-one (train → eval → contact sheet)

Runs the whole edit flow in ONE session, batch-safe for Kaggle **Save Version**:

1. setup (pinned deps, no restart)   2. SD3.5 access + build PIPE eval
3. train edit-LoRA on PIPE pairs **+ zip the adapter immediately**
4. eval in a **fresh subprocess** (avoids the train+reload RAM OOM)
5. show the contact sheet (source | mask | edit)

Learns: fill a masked region so the inserted object matches the photo. Background
is preserved 100% by hard-restore. Trains TWO artifacts (LoRA + input_adapter.pt),
both zipped right after training so they survive even if eval is skipped.

**To run unattended:** set `SMOKE = False` in the train cell, then **Save Version
→ Save & Run All**. Requires GPU + SD3.5 access (HF_TOKEN secret or a mounted
model dataset).

## 1. Setup — install pinned stack (batch-safe, no kernel restart)

Designed to run end-to-end via **Save Version**. The pip install runs before any
`transformers`/`diffusers` import, so a fresh batch kernel loads the pinned
versions directly (no `os._exit` restart, which would abort a batch run). A hard
version assert fails loudly if Kaggle pre-imported a wrong version.

In [ ]:
import subprocess, sys, os
from pathlib import Path

# IMPORTANT (batch-safe): do NOT import transformers/diffusers before this cell.
REPO = Path('/kaggle/working/VIN')
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/BDT-17/VIN.git',str(REPO)], check=True)
else:
    subprocess.run(['git','-C',str(REPO),'fetch','origin'], check=True)
    subprocess.run(['git','-C',str(REPO),'reset','--hard','origin/main'], check=True)
sys.path.insert(0, str(REPO))
print('repo at', subprocess.run(['git','-C',str(REPO),'rev-parse','--short','HEAD'],capture_output=True,text=True).stdout.strip())

# diffusers 0.31.0 needs transformers<=4.46 (FLAX_WEIGHTS_NAME). Install the pinned
# stack first; a fresh batch kernel hasn't imported transformers yet, so these load.
subprocess.run([sys.executable,'-m','pip','install','-q','--force-reinstall','--no-deps',
                'transformers==4.46.3','tokenizers==0.20.3','huggingface_hub==0.25.2'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q',
                'diffusers==0.31.0','accelerate==0.34.2','peft==0.13.2','datasets>=2.20',
                'safetensors>=0.4.3','sentencepiece','protobuf','pillow>=10','numpy'], check=True)

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import transformers, diffusers, torch
print('transformers', transformers.__version__, '| diffusers', diffusers.__version__)
assert transformers.__version__ == '4.46.3', (
    f'transformers is {transformers.__version__}, expected 4.46.3. Kaggle pre-imported a '
    'newer build. Fix: in notebook Settings set "Environment -> Pin to original" OFF / use '
    'the latest env, OR run this cell, then Factory-reset & Run All. For Save Version, ensure '
    'no earlier cell imported transformers.')
from transformers.utils import FLAX_WEIGHTS_NAME  # symbol diffusers 0.31 needs
assert torch.cuda.is_available(), 'No GPU — set Accelerator to GPU'
print('OK on', torch.cuda.get_device_name(0))

## 2. SD3.5 access (gated) + build the PIPE golden eval set

In [ ]:
from pathlib import Path
_local = Path('/kaggle/input/stable-diffusion-3-5-medium')
HF_TOKEN = None
if _local.exists():
    SD35_MODEL = str(_local); print('local SD3.5 mount:', SD35_MODEL)
else:
    SD35_MODEL = 'stabilityai/stable-diffusion-3.5-medium'
    try:
        from kaggle_secrets import UserSecretsClient; HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        import os; HF_TOKEN = os.environ.get('HF_TOKEN')
    assert HF_TOKEN, ('SD3.5 is gated. Add a Kaggle secret HF_TOKEN (Read token + '
                      'agreed access at hf.co/stabilityai/stable-diffusion-3.5-medium) '
                      'or mount the model dataset.')
    from huggingface_hub import login; login(token=HF_TOKEN); print('HF login OK')

In [ ]:
from LoRA.data.build_eval_cases_pipe import run_build_pipe_eval
import json
WORK = Path('/kaggle/working/vin_lora')
EVAL = WORK/'eval'/'pipe_eval_v1'
EVAL_LIMIT = 12   # small for a fast contact sheet; raise for a fuller eval
if not (EVAL/'cases.jsonl').exists():
    run_build_pipe_eval(WORK, eval_set='pipe_eval_v1', split='test', person_only=True, limit=EVAL_LIMIT)
cases = [json.loads(l) for l in (EVAL/'cases.jsonl').read_text().splitlines() if l.strip()]
print(len(cases), 'eval cases')

## 3. Train the edit-LoRA on PIPE pairs

Smoke first (200 steps / 200 samples). For a real adapter set `SMOKE = False`
(uses the config: 1000 steps / 4000 samples — much longer).

In [ ]:
from LoRA.train.train_inpaint_edit import run_training
import shutil
SMOKE = True   # True = 200 steps/200 samples (fast smoke); False = full (3000/4000)
kw = dict(max_train_steps=200, num_train_samples=200) if SMOKE else {}
train = run_training(WORK, base_model_id=SD35_MODEL, hf_token=HF_TOKEN, **kw)
RUN_DIR = train['run_dir']
print('trained ->', RUN_DIR)
# Zip the artifact IMMEDIATELY so it is safe even if a later step crashes.
ZIP = shutil.make_archive(str(RUN_DIR), 'zip', str(RUN_DIR))
print('adapter zip ->', ZIP, '(download this / make it a dataset)')
prov = train['provenance']

# Free the training pipeline's VRAM BEFORE the eval subprocess. run_training already
# drops its GPU refs, but make the kernel's release explicit so the subprocess isn't
# starved (the OOM we hit before: main kernel held ~6GB -> eval had no room).
import gc, torch
del train
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache(); torch.cuda.synchronize()
    free_mb = (torch.cuda.get_device_properties(0).total_memory
               - torch.cuda.memory_reserved(0)) / 1024**2
    print(f'VRAM free after train cleanup: {free_mb:.0f} MB (want most of ~15000)')
prov

## 4. Eval in a FRESH process (avoids train+reload RAM OOM)

Training leaves ~10GB of pipeline + cached tensors in this kernel; loading the
SD3.5 pipeline a second time here would OOM the kernel. So run the edit eval as a
separate process — it starts clean and only holds the inference pipeline. The
adapter is already zipped (cell 3), so this step is optional / re-runnable.

In [ ]:
import subprocess, sys
# Run eval in a clean subprocess (no leftover training RAM -> no reload OOM).
cmd = [sys.executable, '-m', 'LoRA.inference.run_edit_eval',
       '--run-dir', str(RUN_DIR), '--eval-dir', str(EVAL),
       '--base-model', str(SD35_MODEL), '--steps', '30', '--limit', '6']
if HF_TOKEN:
    cmd += ['--hf-token', HF_TOKEN]
print('running eval subprocess...', flush=True)
# capture so the subprocess traceback is visible in the notebook (not swallowed)
r = subprocess.run(cmd, cwd='/kaggle/working/VIN', capture_output=True, text=True)
print('eval exit code:', r.returncode)
print('--- STDOUT (tail) ---'); print(r.stdout[-2500:])
if r.returncode != 0:
    print('--- STDERR (tail) ---'); print(r.stderr[-5000:])
EVAL_OUT = RUN_DIR / 'edit_eval'

## 5. Show the contact sheet (source | mask | edit)

In [ ]:
from pathlib import Path
sheet = EVAL_OUT / 'contact_sheet.png'
assert sheet.exists(), f'eval did not produce a contact sheet (check the subprocess output above): {sheet}'
print('contact sheet:', sheet)
print('metrics csv  :', EVAL_OUT / 'edit_metrics.csv')
from IPython.display import Image as IPImage, display
display(IPImage(str(sheet)))